# Study 922 — Floating-Rate Front End 🪙

**Through hikes and cuts, which end of the cash curve actually pays more?**

A **Treasury floating-rate note** pays a coupon that resets every week off the 13-week
bill auction, plus a small fixed spread. Because the coupon chases the rate, the price
barely moves — the note has almost no *duration*. The pitch writes itself: cash, with a
pickup, and none of the mark-to-market pain when the Fed hikes.

The counter-pitch is just as loud on the way down: a floater collects no term premium and
gets no capital gain when yields fall — the thing that makes a short *fixed* bond fund
worth owning into a cutting cycle.

We race **USFR** and **TFLO** (the two floaters) against **BIL** (1-3 month bills) and
**SHY** (1-3 year fixed) on daily **total-return** closes, 2014-02-04 → 2026-06-30
(3,118 days) — the first complete hike-plateau-cut cycle the Treasury FRN market
has ever lived through.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `4bfc2a85745c`); the
live cells run the fast synthetic control. As-of 2026-06-30.*


## 1. Twelve years, four sleeves, one answer that keeps changing

Held from 2014 to 2026, all four paid roughly the same — a bit under 2% a year, because most of that stretch was spent at zero. The floaters edged it. But the *averages* hide the story: which sleeve you wanted depended entirely on what rates were doing.

In [1]:
R = dict(usfr=1.91, tflo=1.95, bil=1.75, shy=1.48,
         usfr_dd=-2.12, tflo_dd=-5.01, bil_dd=-0.24, shy_dd=-5.71)
for k in ('usfr', 'tflo', 'bil', 'shy'):
    print('%-5s total return %+.2f%%/yr   worst drawdown %+.2f%%'
          % (k.upper(), R[k], R[k + '_dd']))

USFR  total return +1.91%/yr   worst drawdown -2.12%
TFLO  total return +1.95%/yr   worst drawdown -5.01%
BIL   total return +1.75%/yr   worst drawdown -0.24%
SHY   total return +1.48%/yr   worst drawdown -5.71%


## 2. 2022 was the floater's year — and only the floater's year

Split the tape by what the Fed was doing. Through the **hiking** window (2022-03 → 2023-07, 356 trading days) the floaters made **+3.39%/yr** while the 1-3 year fixed fund *lost* **-1.07%/yr** — a gap of **+4.46 pp/yr**. That is not clever: it is arithmetic. A fund with ~1.85 years of duration loses about 1.85% of price for every 1 pp yields rise, and the front end rose 4.94 pp.

Then look at the other windows. In the long **zero-rate** era the fixed fund won (+1.11% vs +0.73%/yr). On the **plateau** it won again (+5.50% vs +4.83%). Two windows each way.

In [2]:
rows = [('ZIRP 2014-2021', 0.73, 0.58, 1.11), ('hiking 2022-23', 3.39, 2.85, -1.07), ('plateau 2023-24', 4.83, 5.24, 5.5), ('cutting 2024-26', 4.24, 4.07, 3.39)]
print(f"{'window':16s} {'USFR':>8s} {'BIL':>8s} {'SHY':>8s}   winner")
for w, u, b, s in rows:
    best = max((u, 'USFR'), (b, 'BIL'), (s, 'SHY'))[1]
    print(f'{w:16s} {u:+7.2f}% {b:+7.2f}% {s:+7.2f}%   {best}')

window               USFR      BIL      SHY   winner
ZIRP 2014-2021     +0.73%   +0.58%   +1.11%   SHY
hiking 2022-23     +3.39%   +2.85%   -1.07%   USFR
plateau 2023-24    +4.83%   +5.24%   +5.50%   SHY
cutting 2024-26    +4.24%   +4.07%   +3.39%   USFR


## 3. The surprise: the cutting cycle never paid duration back

The textbook says the fixed fund gets its revenge when rates fall — its price rises as yields drop. Duration predicted SHY should beat the floater by about **2.3%** cumulatively over 2024-09 → 2026-06.

It didn't. The floater still won by **+1.68%**. Why? Because the curve stayed flat-to-inverted: the 13-week rate fell 1.24 pp but the 1-3 year yield fell far less, and SHY was starting from a *lower* yield than cash in the first place. You paid for duration and the cheque has not arrived.

> 🔬 **For the quants:** this is the time-varying term premium (Fama-Bliss, Cochrane-Piazzesi) in its most concrete form. The forward-implied compensation for two years of duration was negative for most of 2023-2025, so the 'insurance' leg had negative carry going in.

## 4. What is actually reliable here

Two things, and neither is a money machine.

**The pickup over bills is real but tiny.** USFR out-earned BIL by **+0.160%/yr** and TFLO by **+0.198%/yr** — which is just the floating note's spread over the bill rate, net of a 15 bp fee. The point estimate is the same in 2014-2017 and 2018-2026. It is the size of a fee decision, not an edge, and its *t*-statistic (+0.68) says so.

**The drawdown difference is enormous.** Since 2018, the floaters' worst loss was **-0.40%** (USFR) and **-0.16%** (TFLO). SHY's was **-5.71%**, in October 2022. Same return, a fourteenth of the pain — that is the real reason to own the floating end of the curve.

## 5. Why we won't call it Real

Everything above points the same way, and the effect is big. But **twelve years contains exactly one hiking cycle**, and one macro event is one observation however many trading days it spans. The single formal test — does the direction of rates flip the ranking? — comes back at **+2.43 pp/yr with *t* = +1.37**, short of the desk's |*t*| = 2 bar. The bootstrap interval on the floater-over-bills pickup, [-0.244, +0.548], contains zero.

> 🔬 **For the quants:** we lag every ^IRX-derived object exactly one day, use Newey-West errors and a 21-day block bootstrap precisely so the 626 'rising' days cannot masquerade as 626 independent draws.

## 6. Live check — the machinery is unbiased (offline synthetic)

Two toy worlds with the *same* rate cycle. In the first, the fixed leg has a real 1.85-year duration, so rate direction must flip the ranking. In the second, the 'fixed' leg is secretly a floater — rates still swing, but there is nothing to find. A trustworthy estimator shouts in the first and stays silent in the second.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from frn_front import data, strategy as st
planted = st.synthetic_detect(data.synthetic_panel(signal_strength=1.0, seed=922)[0])
null    = st.synthetic_detect(data.synthetic_panel(signal_strength=0.0, seed=922)[0])
print('fixed leg HAS duration : rate-direction contrast %+.2f pp/yr (t=%+.1f)'
      % (planted['contrast'], planted['contrast_t']))
print('fixed leg has NONE     : rate-direction contrast %+.2f pp/yr (t=%+.1f)'
      % (null['contrast'], null['contrast_t']))
print('(same rate cycle both times: %d rising days, %d falling days)'
      % (null['n_rising'], null['n_falling']))
import numpy as np
nl = np.array([st.synthetic_detect(data.synthetic_panel(signal_strength=0.0, seed=922+s)[0])['contrast']
               for s in range(8)])
print('across 8 null worlds   : %+.2f pp/yr on average (spread %.2f) - noise, not signal'
      % (nl.mean(), nl.std(ddof=1)))

fixed leg HAS duration : rate-direction contrast +8.35 pp/yr (t=+14.2)
fixed leg has NONE     : rate-direction contrast +0.69 pp/yr (t=+1.3)
(same rate cycle both times: 416 rising days, 947 falling days)


across 8 null worlds   : +0.14 pp/yr on average (spread 0.40) - noise, not signal


## Verdict

- **Signal — Weak.** The floating end really did out-pay the fixed end over the cycle, and the ranking really does flip with the direction of rates — but the whole margin was earned in one 17-month window, and **no test anywhere in the study clears |*t*| = 2** (headline contrast +2.43 pp/yr, *t* = +1.37; 0 of 12 classifier settings). Textbook mechanism, uncertified tape.
- **Tradability — Fragile.** ~15 bps/yr over bills is a fee-sized decision; beating SHY requires knowing which way rates are going. What you *can* bank, for free, is the risk profile: -0.4% worst drawdown against -5.7%. Choose your duration deliberately — don't expect the floater to be alpha.